-----

#### **Python Script for Synthetic Data Generation**

In [2]:
# Install the Faker library if it's not already installed
!pip install Faker pandas

# Data Dictionary: Latto Lux Marketing Campaign

This document provides a detailed description of all data fields used in the "Latto Lux" data science project.

---

### **Dataset 1: `physical_sales`**

This dataset represents transactions of the physical "Latto Lux" scratch-off tickets.

| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `transaction_id` | STRING | A unique identifier for each individual ticket sale. |
| `date` | DATE | The date of the sale. |
| `time` | TIME | The time of the sale. |
| `retailer_location_city` | STRING | The city where the ticket was purchased (e.g., 'Atlanta', 'Chicago', 'New York'). |
| `ticket_batch_id` | STRING | An identifier for the batch of tickets the item belongs to. |
| `unique_code` | STRING | The unique alphanumeric code printed on the ticket. This code is the link to the digital app usage data. |

---

### **Dataset 2: `digital_app_usage`**

This dataset tracks user engagement and activity within the "Latto Lux" mobile application.

| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `user_id` | STRING | A unique, anonymized identifier for each user of the app. |
| `date` | DATE | The date of the app session. |
| `time` | TIME | The time of the app session. |
| `app_session_length_minutes` | INTEGER | The duration of the user's app session in minutes. |
| `feature_used` | STRING | The specific app feature the user engaged with (e.g., 'scan_ticket', 'play_ar_game', 'share_social'). |
| `unique_code` | STRING | The alphanumeric code from the physical ticket that the user scanned. |
| `social_share_count` | INTEGER | The number of times the user shared content to social media from within the app. |

In [5]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

# Instantiate the Faker generator
fake = Faker()

# Define campaign start and end dates for a 6-month period
start_date = datetime(2025, 8, 23)
end_date = start_date + timedelta(days=180)

# List of key Latto markets
latto_markets = ['Atlanta', 'New York', 'Los Angeles', 'Chicago', 'Houston', 'Miami', 'Detroit']

#
# 1. Generate the physical_sales.csv dataset
#

print("Generating physical_sales.csv...")

def generate_physical_sales(num_records=20000):
    """
    Generates a synthetic dataset for physical scratch-off ticket sales.
    """
    data = []
    # Generate unique codes first to ensure they can be linked to the app data
    unique_codes = [fake.bothify(text='????-####', letters='ABCDEFGHIJKLMNOPQRSTUVWXYZ') for _ in range(num_records)]

    for i in range(num_records):
        date = fake.date_time_between(start_date=start_date, end_date=end_date)
        row = {
            'transaction_id': f'T{i+1:06}',
            'date': date.date(),
            'time': date.time().replace(microsecond=0), # Remove microseconds
            'retailer_location_city': random.choice(latto_markets),
            'ticket_batch_id': random.choice(['BATCH-A-2025', 'BATCH-B-2025', 'BATCH-C-2025']),
            'unique_code': unique_codes[i]
        }
        data.append(row)

    df = pd.DataFrame(data)
    # Ensure date and time columns are in a format BigQuery can handle
    df['date'] = pd.to_datetime(df['date']).dt.date
    df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S').dt.time

    print("Saving physical_sales.csv...")
    df.to_csv('physical_sales.csv', index=False)
    return df['unique_code'].tolist() # Return the list of codes for the next dataset

#
# 2. Generate the digital_app_usage.csv dataset
#

print("Generating digital_app_usage.csv...")

def generate_digital_app_usage(unique_codes, num_records=50000):
    """
    Generates a synthetic dataset for digital app usage.
    """
    data = []
    # Create a pool of users to simulate repeat visits
    user_ids = [f'U{i+1:05}' for i in range(5000)]

    # Randomly select a subset of physical unique codes to be "activated" in the app
    # This simulates not every ticket being scanned
    activated_codes = random.sample(unique_codes, k=int(len(unique_codes) * 0.75))

    for i in range(num_records):
        date = fake.date_time_between(start_date=start_date, end_date=end_date)

        # Simulate linking a physical ticket to a digital user
        # 80% of sessions are linked to a scanned ticket, 20% are not (e.g., browsing)
        if random.random() < 0.8:
            code_link = random.choice(activated_codes)
        else:
            code_link = None

        row = {
            'user_id': random.choice(user_ids),
            'date': date.date(),
            'time': date.time().replace(microsecond=0), # Remove microseconds
            'app_session_length_minutes': random.randint(1, 30),
            'feature_used': random.choice(['scan_ticket', 'play_ar_game', 'share_social', 'browse_merch', 'stream_music']),
            'unique_code': code_link,
            'social_share_count': random.randint(0, 5) if random.random() < 0.3 else 0 # 30% of users share content
        }
        data.append(row)

    df = pd.DataFrame(data)
    # Ensure date and time columns are in a format BigQuery can handle
    df['date'] = pd.to_datetime(df['date']).dt.date
    df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S').dt.time

    print("Saving digital_app_usage.csv...")
    df.to_csv('digital_app_usage.csv', index=False)

# Run the generation functions
all_unique_codes = generate_physical_sales()
generate_digital_app_usage(all_unique_codes)

print("\nSynthetic data generation complete!")
print("The files 'physical_sales.csv' and 'digital_app_usage.csv' have been created.")

Generating physical_sales.csv...
Generating digital_app_usage.csv...
Saving physical_sales.csv...
Saving digital_app_usage.csv...

Synthetic data generation complete!
The files 'physical_sales.csv' and 'digital_app_usage.csv' have been created.
